In [1]:
## from transformers import AutoModelForDepthEstimation
import sys
import kornia
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from PIL import Image
from IPython.display import display
import os
import random
import numpy as np
import multiprocessing
import subprocess
torch.set_float32_matmul_precision('medium')
import torch.distributed as dist
from elasticdino.model.elasticdino import ElasticDino
depth_anything_path = "/mnt/home/mizrahiulysse/model_cache/depth_anything_v2_large/"



%reload_ext tensorboard



In [2]:

# HYPERSIM_PATHS = []
# N_HYPERSIM_IMAGES = 6
# HYPERSIM_BASE_PATH = "/mnt/home/mizrahiulysse/datasets/hypersim-depth/"

# for path in os.listdir(HYPERSIM_BASE_PATH):
#     for subpath in os.listdir(os.path.join(HYPERSIM_BASE_PATH, path, "images")):
#         frames = os.listdir(os.path.join(HYPERSIM_BASE_PATH, path, "images", subpath))
#         print(frames)
#         raise
#         frames = [x for x in frames if "color" in x]
#         if len(frames) >= 6:
#             HYPERSIM_PATHS.append(os.path.join(HYPERSIM_BASE_PATH, path, "images", subpath))

# train_size = int(len(HYPERSIM_PATHS) * TRAIN_PROPORTION)
# HYPERSIM_TRAIN_PATHS = HYPERSIM_PATHS[:train_size]
# HYPERSIM_TEST_PATHS = HYPERSIM_PATHS[train_size:]

# def process_image(img):
#     l = min(img.height, img.width)
#     return img.crop((0, 0, l, l)).resize((IMAGE_SIZE, IMAGE_SIZE))


# def perturb_images(imgs):
#     # hue
#     values = torch.rand(imgs.shape[0])
#     values = (values - 0.5) * 0.3
#     imgs = kornia.enhance.adjust_hue(imgs, values)

#     # gamma
#     values = torch.rand(imgs.shape[0])
#     values = 1 + (values - 0.5)
#     imgs = kornia.enhance.adjust_gamma(imgs, values)
#     return imgs
    
# def hypersim_sample(folder):
#     imgs = []
#     albedos = []
#     paths = os.listdir(folder)
#     frames = [x for x in set([x.split(".")[1] for x in paths])]
#     frames = sorted(frames)[:N_HYPERSIM_IMAGES]    
#     for f in frames:
#         img = process_image(Image.open(os.path.join(folder, f"frame.{f}.color.jpg")))
#         albedo = process_image(Image.open(os.path.join(folder, f"frame.{f}.diffuse_reflectance.jpg")))
#         img =  torchvision.transforms.functional.pil_to_tensor(img)/255.0
#         albedo = torchvision.transforms.functional.pil_to_tensor(albedo)/255.0
#         imgs.append(img)
#         albedos.append(albedo)
#     imgs = torch.stack(imgs)
#     albedos = torch.stack(albedos)
#     masks = torch.ones_like(imgs) > 0
#     imgs = perturb_images(imgs)
#     return imgs, albedos, masks
    

# class HypersimDataset(torch.utils.data.Dataset):
#     def __init__(self, paths):
#         self.paths = paths

#     def __len__(self):
#         return len(self.paths)

#     def __getitem__(self, idx):
#         path = self.paths[idx]
#         return hypersim_sample(path)


# hypersim_train_ds = HypersimDataset(HYPERSIM_TRAIN_PATHS)
# hypersim_test_ds = HypersimDataset(HYPERSIM_TEST_PATHS)

In [3]:
import numpy as np
import torchvision
import torch

nyu_val_path = "/mnt/home/mizrahiulysse/datasets/nyu-val"

    

class NyuDataset(torch.utils.data.Dataset):
    def __init__(self, base_path, image_size):
        self.paths = os.listdir(os.path.join(nyu_val_path, "rgb"))
        self.base_path = base_path
        self.image_size = image_size

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        rgb = np.load(os.path.join(self.base_path, "rgb", path))[:, :480, :480] / 255.0
        depth = np.load(os.path.join(self.base_path, "depth", path))[:480, :480]
        rgb = torch.nn.functional.interpolate(torch.from_numpy(rgb).unsqueeze(0), self.image_size).squeeze(0)
        depth = torch.nn.functional.interpolate(torch.from_numpy(depth).unsqueeze(0).unsqueeze(0), self.image_size).squeeze(0)
        masks = torch.ones_like(depth).to(dtype=torch.bool)
        return rgb, depth, masks

def get_nyu_dataloader(base_path, batch_size, image_size, shuffle):
    dataset = NyuDataset(base_path, image_size)
    return torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=16)

# for x,y, z in get_nyu_dataloader(nyu_val_path, 4, 128, False):
#     print(x, y, z)
#     break

In [4]:
diode_base_path = "/mnt/home/mizrahiulysse/datasets/diode-val/val/"


class DiodeDataset(torch.utils.data.Dataset):
    def __init__(self, base_path, image_size):
        self.truncated_paths = []
        self.image_size = image_size
        for root, dirs, files in os.walk(diode_base_path):
            for file in files:
              if "png" in file.lower():
                image_id = file.split(".")[0]
                self.truncated_paths.append(os.path.join(root, image_id))
                
    def __len__(self):
        return len(self.truncated_paths)

    def __getitem__(self, idx):
        truncated_path = self.truncated_paths[idx]

        rgb = Image.open(f"{truncated_path}.png").crop((0, 0, 768, 768)).resize((self.image_size, self.image_size))
        rgb = torchvision.transforms.functional.pil_to_tensor(rgb) / 255.0
        
        depth = np.load(f"{truncated_path}_depth.npy")[:768, :768]
        depth_mask = np.load(f"{truncated_path}_depth_mask.npy")[:768, :768]
                
        depth = torch.nn.functional.interpolate(torch.from_numpy(depth).squeeze().unsqueeze(0).unsqueeze(0), self.image_size, mode="bilinear").squeeze(0)
        depth_mask = torch.nn.functional.interpolate(torch.from_numpy(depth_mask).squeeze().unsqueeze(0).unsqueeze(0), self.image_size, mode="nearest").squeeze(0)
        
        return rgb, depth, depth_mask > 0

def get_diode_dataloader(base_path, batch_size, image_size, shuffle):
    dataset = DiodeDataset(base_path, image_size)
    return torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=16)


In [14]:
from accelerate import Accelerator
from accelerate.utils import set_seed, DistributedDataParallelKwargs
from elasticdino.training.depth.layers import DPTDepthModel, ElasticDinoDepthModel, UNet
from elasticdino.model.dino import DinoV2

# kwargs = DistributedDataParallelKwargs(find_unused_parameters=True)
dynamo_backend = "inductor" if False else "no"
accelerator = Accelerator(mixed_precision="fp16", dynamo_backend=dynamo_backend)

ed = ElasticDino.from_pretrained("/mnt/home/mizrahiulysse/pixelvit-32-L.pth", "elasticdino-32-L")
model = ElasticDinoDepthModel(512, ed)
# dino = DinoV2("l")
# model = DPTDepthModel(512, dino, 128)
model = accelerator.prepare(model)
accelerator.load_state("/mnt/home/mizrahiulysse/elasticdino-runs/pretrain-elasticdino-depth/ED/2025-01-28-04:10:24/checkpoints/80000")
# accelerator.load_state("/mnt/home/mizrahiulysse/elasticdino-runs/pretrain-elasticdino-depth/DPT/2025-01-28-04:43:49/checkpoints/80000")

DA_CACHE = {}


Using cache found in /mnt/home/mizrahiulysse/model_cache/torch/hub/facebookresearch_dinov2_main
INFO:dinov2:using MLP layer as FFN
INFO:accelerate.accelerator:Loading states from /mnt/home/mizrahiulysse/elasticdino-runs/pretrain-elasticdino-depth/ED/2025-01-28-04:10:24/checkpoints/80000
INFO:accelerate.checkpointing:All model weights loaded successfully
INFO:accelerate.checkpointing:All optimizer states loaded successfully
INFO:accelerate.checkpointing:All scheduler states loaded successfully
INFO:accelerate.checkpointing:All dataloader sampler states loaded successfully
INFO:accelerate.checkpointing:GradScaler state loaded successfully
INFO:accelerate.checkpointing:All random states loaded successfully
INFO:accelerate.accelerator:Loading in 0 custom states


In [11]:

from transformers import AutoModelForDepthEstimation
depth_anything_path = "/mnt/home/mizrahiulysse/model_cache/depth_anything_v2_large/"

depth_size = 518

def preprocess_image_for_depth(image_tensor):
  depth_image_mean = torch.tensor([
        0.485,
        0.456,
        0.406
      ]).reshape((1, 3, 1, 1)).to(device=image_tensor.device)
    
  depth_image_std = torch.tensor([
        0.229,
        0.224,
        0.225
      ]).reshape((1, 3, 1, 1)).to(device=image_tensor.device)
  image_tensor = (image_tensor - depth_image_mean) / depth_image_std
  image_tensor = torch.nn.functional.interpolate(
      image_tensor,
      size=(depth_size, depth_size),
      mode="bilinear",
      align_corners=False,
      antialias=True
  )
  return image_tensor


@torch.no_grad()
def get_depth_anything_depth(depth_anything, images, normalize=False):
  size = images.shape[-1]
  with torch.no_grad():
    inputs = preprocess_image_for_depth(images)
    outputs = depth_anything(pixel_values=inputs)
    predicted_depth = outputs.predicted_depth
    predicted_depth = torch.nn.functional.interpolate(
        predicted_depth.unsqueeze(1),
        size=(size, size),
        mode="bilinear",
        align_corners=False,
        antialias=True
    )
    if normalize:
        predicted_depth[torch.isnan(predicted_depth)] = 0
        m = predicted_depth.min(dim=2, keepdim=True)[0].min(dim=3, keepdim=True)[0]
        M = predicted_depth.max(dim=2, keepdim=True)[0].max(dim=3, keepdim=True)[0]
        predicted_depth = (predicted_depth - m) / (M - m + 1e-5)
    return predicted_depth

depth_anything_path = "/mnt/home/mizrahiulysse/model_cache/depth_anything_v2_large/"

# print("load")
# with torch.no_grad():
#     depth_anything = AutoModelForDepthEstimation.from_pretrained(depth_anything_path, local_files_only=True)



def make_pretraining_dataloader(dataloader):
  if depth_anything_path in DA_CACHE:
    depth_anything = DA_CACHE[depth_anything_path]
  else:
    with torch.no_grad():
      print("loading depth anything")
      depth_anything = AutoModelForDepthEstimation.from_pretrained(depth_anything_path, local_files_only=True).to(device="cuda")
      depth_anything = torch.compile(depth_anything)
      DA_CACHE[depth_anything_path] = depth_anything
    
  for images, depth, masks in dataloader:
    images = images.to(device="cuda", non_blocking=False)
    depth = depth.to(device="cuda", non_blocking=False)
    masks = masks.to(device="cuda", non_blocking=False)
    da = get_depth_anything_depth(depth_anything, images)
    yield images, depth, masks, da

In [15]:
import matplotlib.pyplot as plt

dataloader = make_pretraining_dataloader(get_diode_dataloader(diode_base_path, 4, 128, False))
dataloader = accelerator.prepare(dataloader)


    
def normalize(x, mask):
    # g = x[mask].log().mean(dim=(1, 2), keepdim=True).exp()
    logs = torch.where(mask, x.log(), 0.0).sum(dim=(2, 3), keepdim=True)
    counts = mask.sum(dim=(2, 3), keepdim=True) 
    g = (logs / counts).exp()
    return x / g

def normalize_invariant(x, mask):
    B, _, H, W = x.shape
    x = x.reshape(B, H * W)
    mask = mask.reshape(B, H * W)
    median = torch.nanmedian(torch.where(mask, x, torch.nan), dim=1, keepdim=True).values
    x = x - median
    s = torch.sum(torch.where(mask, x.abs(), 0), dim=1, keepdim=True)
    c = torch.sum(mask, dim=1, keepdim=True)
    s = s / c    
    x = x / (s + 1e-6)
    mask = mask & ~torch.isnan(x)
    x[~mask] = 0.0
    return x.reshape((B, 1, H, W))
    
    
def diode_to_disparity(depth, mask):
    depth[depth < 1] = 1
    # depth[depth > 10] = 80
    # depth = normalize(depth, mask)
    res = 1.0 / depth
    res[~mask] = 1
    
    return res
    
def plot(x, mask):
    data = x[0][mask[0]].flatten().cpu().numpy()
    print(np.max(data))

    # Plot the histogram
    plt.hist(data, bins=30, edgecolor='black', alpha=0.7)
    plt.grid(axis='y', linestyle='-', color='lightgray')
    plt.ylabel('Frequency')
    plt.xlabel('Value')
    plt.title('Distribution of Tensor Values')
    plt.show()
    
def display_depth(x, mask=None):
    x = x[0][0]
    # if mask is not None:
    #     mask = mask[0][0]
    #     x[~mask] = 1
    x = torch.log(1+x)
    print(torch.max(x), torch.min(x))
    x = 255 * (x - torch.min(x)) / (torch.max(x) - torch.min(x))
    display(Image.fromarray(x.detach().cpu().numpy().astype(np.uint8)))

with torch.no_grad():
    mae = 0.0
    mae_da = 0.0
    n = 0
    for rgb, depth, mask, da in dataloader:
        da = da.clamp(1e-3)
        with accelerator.autocast():
            mask = mask & (depth > 1e-3) & (depth < 20)
            res = model(rgb)
            da = normalize_invariant(da, mask)
            depth = normalize_invariant(1/depth.clamp(1e-3, 20), mask)
            res = normalize_invariant(res, mask)
            mae += (res - depth).abs().mean()
            mae_da += (da - depth).abs().mean()
            n += 1
            print(mae /n, mae_da/n)
    print(mae / n, mae_da / n)

loading depth anything
tensor(0.3207, device='cuda:0') tensor(0.2311, device='cuda:0')
tensor(0.5295, device='cuda:0') tensor(0.3865, device='cuda:0')
tensor(0.5862, device='cuda:0') tensor(0.4442, device='cuda:0')
tensor(0.5775, device='cuda:0') tensor(0.4311, device='cuda:0')
tensor(0.5747, device='cuda:0') tensor(0.4441, device='cuda:0')
tensor(0.5357, device='cuda:0') tensor(0.4041, device='cuda:0')
tensor(0.5136, device='cuda:0') tensor(0.3871, device='cuda:0')
tensor(0.5073, device='cuda:0') tensor(0.3848, device='cuda:0')
tensor(0.5035, device='cuda:0') tensor(0.3795, device='cuda:0')
tensor(0.5093, device='cuda:0') tensor(0.3744, device='cuda:0')
tensor(0.5121, device='cuda:0') tensor(0.3744, device='cuda:0')
tensor(0.5227, device='cuda:0') tensor(0.3796, device='cuda:0')
tensor(0.5275, device='cuda:0') tensor(0.3794, device='cuda:0')
tensor(0.5196, device='cuda:0') tensor(0.3761, device='cuda:0')
tensor(0.5006, device='cuda:0') tensor(0.3632, device='cuda:0')
tensor(0.5205, de